In [ ]:
# ==============================================
# 1. INSPECCION INICIAL DE LOS DATOS
# ==============================================

import pandas as pd
import numpy as np

df = pd.read_csv('logistica_banano_uraba.csv')

print('PRIMERAS 5 FILAS')
display(df.head())

print('\nDIMENSIONES DEL DATASET')
print(f'Registros: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

print('\nMUNICIPIOS IDENTIFICADOS')
print(df['municipio_origen'].unique())

In [ ]:
# ==============================================
# 2. LIMPIEZA Y ESTANDARIZACION
# ==============================================

# Normalizar los nombres de los municipios
df['municipio_origen'] = df['municipio_origen'].str.strip().str.title()

correcciones = {
    'Apartado': 'Apartadó',
    'Chigorodo': 'Chigorodó'
}

df['municipio_origen'] = df['municipio_origen'].replace(correcciones)

# Convertir la fecha al formato datetime
df['fecha_envio'] = pd.to_datetime(df['fecha_envio'], errors='coerce')

# Conservar los valores faltantes de horas de tránsito
# para analizarlos antes de decidir cómo imputarlos.

print('MUNICIPIOS DESPUES DE LA ESTANDARIZACION')
print(df['municipio_origen'].unique())

print('\nVALORES FALTANTES DESPUES DE LA LIMPIEZA INICIAL')
print(df.isnull().sum())

In [ ]:
# ==============================================
# 3. DIAGNOSTICO DE CALIDAD DE LOS DATOS
# ==============================================

print('DIMENSIONES DEL DATASET')
print(f'Registros: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

print('\nTIPOS DE DATOS')
print(df.dtypes)

print('\nVALORES FALTANTES')
faltantes = df.isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

diagnostico = pd.DataFrame({
    'Valores faltantes': faltantes,
    'Porcentaje (%)': porcentaje_faltantes.round(2)
})

display(diagnostico)

print('\nREGISTROS DUPLICADOS')
print(df.duplicated().sum())

print('\nESTADISTICAS DESCRIPTIVAS')
display(df.describe().round(2))

## 4. Analisis exploratorio de datos (EDA)

En esta etapa se estudiaran las variables del conjunto de datos para comprender su comportamiento, identificar patrones y detectar posibles relaciones entre las condiciones logisticas y el rechazo de cajas.

Los datos utilizados son sinteticos, por lo que los resultados representan el comportamiento del escenario simulado y no necesariamente la realidad de las exportaciones de banano.

In [ ]:
# ==============================================
# 5. ANALISIS DE VARIABLES NUMERICAS
# ==============================================

columnas_numericas = [
    'horas_transito',
    'cajas_enviadas',
    'cajas_aprobadas',
    'cajas_rechazadas'
]

print('RESUMEN ESTADISTICO')
display(df[columnas_numericas].describe().round(2))

print('\nPORCENTAJE DE RECHAZO GENERAL')
total_enviadas = df['cajas_enviadas'].sum()
total_rechazadas = df['cajas_rechazadas'].sum()

porcentaje_rechazo = (total_rechazadas / total_enviadas) * 100

print(f'Porcentaje de rechazo: {porcentaje_rechazo:.2f}%')

In [ ]:
# ==============================================
# 6. ANALISIS POR MUNICIPIO
# ==============================================

analisis_municipio = df.groupby('municipio_origen').agg(
    cajas_enviadas=('cajas_enviadas', 'sum'),
    cajas_rechazadas=('cajas_rechazadas', 'sum'),
    viajes=('id_viaje', 'count')
).reset_index()

analisis_municipio['porcentaje_rechazo'] = (
    analisis_municipio['cajas_rechazadas'] /
    analisis_municipio['cajas_enviadas'] * 100
)

analisis_municipio = analisis_municipio.sort_values(
    by='porcentaje_rechazo', ascending=False
)

display(analisis_municipio.round(2))

In [ ]:
# ==============================================
# 7. ANALISIS POR TURNO DE DESPACHO
# ==============================================

analisis_turno = df.groupby('turno').agg(
    cajas_enviadas=('cajas_enviadas', 'sum'),
    cajas_rechazadas=('cajas_rechazadas', 'sum'),
    viajes=('id_viaje', 'count')
).reset_index()

analisis_turno['porcentaje_rechazo'] = (
    analisis_turno['cajas_rechazadas'] /
    analisis_turno['cajas_enviadas'] * 100
)

display(analisis_turno.round(2))

In [ ]:
# ==============================================
# 8. ANALISIS DE TIEMPOS DE TRANSITO
# ==============================================

print('RESUMEN DE HORAS DE TRANSITO')
display(df['horas_transito'].describe().round(2))

print('\nRELACION ENTRE TIEMPO DE TRANSITO Y RECHAZO')

analisis_transito = df.groupby(
    pd.cut(df['horas_transito'], bins=[0, 2, 4, 6, 8])
).agg(
    cajas_enviadas=('cajas_enviadas', 'sum'),
    cajas_rechazadas=('cajas_rechazadas', 'sum'),
    viajes=('id_viaje', 'count')
)

analisis_transito['porcentaje_rechazo'] = (
    analisis_transito['cajas_rechazadas'] /
    analisis_transito['cajas_enviadas'] * 100
)

display(analisis_transito.round(2))

In [ ]:
# ==============================================
# 9. VISUALIZACIONES INICIALES
# ==============================================

import matplotlib.pyplot as plt

# Rechazo por municipio
plt.figure(figsize=(10, 5))
plt.bar(
    analisis_municipio['municipio_origen'],
    analisis_municipio['porcentaje_rechazo']
)
plt.title('Porcentaje de rechazo por municipio')
plt.xlabel('Municipio')
plt.ylabel('Rechazo (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Rechazo por turno
plt.figure(figsize=(10, 5))
plt.bar(
    analisis_turno['turno'],
    analisis_turno['porcentaje_rechazo']
)
plt.title('Porcentaje de rechazo por turno')
plt.xlabel('Turno')
plt.ylabel('Rechazo (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# 10. EXPORTACION DEL DATASET LIMPIO
# ==============================================

# Por ahora se conservan los valores faltantes de horas_transito.
# La imputacion se realizara despues de evaluar su distribucion.

df.to_csv(
    'logistica_limpia.csv',
    index=False,
    encoding='utf-8'
)

print('Archivo logistica_limpia.csv exportado correctamente.')
print(f'Registros exportados: {len(df)}')
print(f'Valores faltantes en horas de transito: {df["horas_transito"].isnull().sum()}')